In [1]:

import pandas as pd
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

In [4]:
!pip install kaggle

from google.colab import files
files.upload()  # upload kaggle.json
import os
os.makedirs('/root/.kaggle', exist_ok=True)
!mv kaggle.json /root/.kaggle/
!chmod 600 /root/.kaggle/kaggle.json
!kaggle datasets download -d luisreimberg/ratingscsv
!unzip ratingscsv.zip

mv: cannot stat 'kaggle.json': No such file or directory
chmod: cannot access '/root/.kaggle/kaggle.json': No such file or directory
Dataset URL: https://www.kaggle.com/datasets/luisreimberg/ratingscsv
License(s): unknown
100% 680k/680k [00:00<00:00, 114MB/s]

Archive:  ratingscsv.zip
  inflating: ratings.csv             


In [7]:
ratings = pd.read_csv("ratings.csv")

In [8]:

# ================= USER-ITEM MATRIX =================
user_item_matrix = ratings.pivot_table(
    index='userId',
    columns='movieId',
    values='rating'
).fillna(0)

In [9]:

# ================= USER SIMILARITY =================
user_similarity = cosine_similarity(user_item_matrix)
user_similarity_df = pd.DataFrame(
    user_similarity,
    index=user_item_matrix.index,
    columns=user_item_matrix.index
)

In [10]:

# ================= RECOMMEND FUNCTION =================
def recommend_movies(user_id, top_n=5):
    if user_id not in user_item_matrix.index:
        return "User not found!"

    # Similar users
    similar_users = user_similarity_df[user_id].sort_values(ascending=False)[1:6]

    # Weighted ratings
    weighted_ratings = np.zeros(user_item_matrix.shape[1])

    for sim_user, score in similar_users.items():
        weighted_ratings += score * user_item_matrix.loc[sim_user].values

    # Normalize
    weighted_ratings /= similar_users.sum()

    # Get unseen movies
    user_rated = user_item_matrix.loc[user_id]
    unseen_movies = user_rated[user_rated == 0].index

    scores = list(zip(unseen_movies, weighted_ratings[user_item_matrix.columns.get_indexer(unseen_movies)]))
    scores = sorted(scores, key=lambda x: x[1], reverse=True)

    recommended_ids = [movie for movie, _ in scores[:top_n]]

    return recommended_ids


In [11]:
print(recommend_movies(user_id=1))

[1200, 1610, 541, 589, 1036]
